In [ ]:
import os
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely import affinity
from pypinyin import lazy_pinyin

# =========================
# 0. Global settings
# =========================
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.sans-serif"] = ["Times New Roman"]
plt.rcParams["font.serif"] = ["Times New Roman"]
plt.rcParams["axes.unicode_minus"] = False

# =========================
# 1. Paths
# =========================
save_dir = r"D:\code\音调识别\0520\complete_package"
os.makedirs(save_dir, exist_ok=True)

CSV_FILE = os.path.join(save_dir, "zuobia_geocoded.csv")
OUT_PNG = os.path.join(save_dir, "map_county_rotated_landscape.png")
OUT_PDF = os.path.join(save_dir, "map_county_rotated_landscape.pdf")

# Prefecture-level city adcodes involved in the study
city_files = {
    "Wuhu": "340200",
    "Ma'anshan": "340500",
    "Tongling": "340700",
    "Huangshan": "341000",
    "Chizhou": "341700",
    "Xuancheng": "341800",
    "Nanjing": "320100",
}

BASE_URL = "https://geo.datav.aliyun.com/areas_v3/bound/{}_full.json"

# 地图旋转角度
# -90：顺时针旋转 90 度
#  90：逆时针旋转 90 度
# 如果你发现方向反了，就把 -90 改成 90
ROTATE_ANGLE = -50

# =========================
# 2. Download county-level boundaries
# =========================
def download_geojson(url, filename):
    if not os.path.exists(filename):
        print(f"Downloading {filename} ...")
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        with open(filename, "wb") as f:
            f.write(r.content)

geo_dfs = []

for city_name, adcode in city_files.items():
    f = os.path.join(save_dir, f"{adcode}_full.json")
    download_geojson(BASE_URL.format(adcode), f)
    g = gpd.read_file(f)
    geo_dfs.append(g)

county_map = pd.concat(geo_dfs, ignore_index=True)
county_map = gpd.GeoDataFrame(county_map, geometry="geometry", crs="EPSG:4326")

# =========================
# 3. Read point data
# =========================
df = pd.read_csv(CSV_FILE, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]

plot_df = df.dropna(subset=["lon", "lat"]).copy()

gdf = gpd.GeoDataFrame(
    plot_df,
    geometry=gpd.points_from_xy(plot_df["lon"], plot_df["lat"]),
    crs="EPSG:4326"
)

# =========================
# 4. Convert Chinese labels to English / Pinyin
# =========================
NAME_OVERRIDES = {
    "马鞍山": "Ma'anshan",
    "黄山": "Huangshan",
    "南京": "Nanjing",
    "宣城": "Xuancheng",
    "芜湖": "Wuhu",
    "铜陵": "Tongling",
    "池州": "Chizhou",
    "绩溪": "Jixi",
    "歙县": "Shexian",
    "黟县": "Yixian",
    "休宁": "Xiuning",
    "泾县": "Jingxian",
    "旌德": "Jingde",
    "青阳": "Qingyang",
    "石台": "Shitai",
    "东至": "Dongzhi",
    "南陵": "Nanling",
    "繁昌": "Fanchang",
    "郎溪": "Langxi",
    "广德": "Guangde",
    "宁国": "Ningguo",
}

def to_english_label(text):
    if pd.isna(text):
        return ""
    text = str(text).strip()
    if text in NAME_OVERRIDES:
        return NAME_OVERRIDES[text]
    return "".join(lazy_pinyin(text)).title()

plot_df["name_en"] = plot_df["name"].apply(to_english_label)
gdf["name_en"] = plot_df["name_en"].values

# =========================
# 5. Rotate map and points together
# =========================
# 关键：所有县界和点位都必须围绕同一个中心旋转
xmin, ymin, xmax, ymax = county_map.total_bounds
cx = (xmin + xmax) / 2
cy = (ymin + ymax) / 2
rotate_origin = (cx, cy)

county_map_rot = county_map.copy()
county_map_rot["geometry"] = county_map_rot["geometry"].apply(
    lambda geom: affinity.rotate(
        geom,
        ROTATE_ANGLE,
        origin=rotate_origin,
        use_radians=False
    )
)

gdf_rot = gdf.copy()
gdf_rot["geometry"] = gdf_rot["geometry"].apply(
    lambda geom: affinity.rotate(
        geom,
        ROTATE_ANGLE,
        origin=rotate_origin,
        use_radians=False
    )
)

gdf_rot["x_rot"] = gdf_rot.geometry.x
gdf_rot["y_rot"] = gdf_rot.geometry.y

# =========================
# 6. Plot style
# =========================
fig, ax = plt.subplots(figsize=(14, 8), facecolor="white")
ax.set_facecolor("white")

# County polygons + county boundaries
county_map_rot.plot(
    ax=ax,
    facecolor="none",
    edgecolor="#4A90E2",
    linewidth=0.7,
    zorder=1
)

# Thicker outer boundary
outline = county_map_rot.dissolve()
outline.plot(
    ax=ax,
    facecolor="none",
    edgecolor="#2F6FD6",
    linewidth=1.2,
    zorder=2
)

# =========================
# 7. Plot rotated points
# =========================
style_map = {
    "main": {"color": "red", "marker": "*", "size": 150, "zorder": 5},
    "sub": {"color": "dodgerblue", "marker": "o", "size": 30, "zorder": 4},
    "sub2": {"color": "limegreen", "marker": "^", "size": 40, "zorder": 4},
}

label_map = {
    "main": "Key Survey Area",
    "sub": "Survey Site",
    "sub2": "General Research Area",
}

for t, group in gdf_rot.groupby("type"):
    st = style_map.get(
        t,
        {"color": "black", "marker": "o", "size": 20, "zorder": 3}
    )

    ax.scatter(
        group["x_rot"],
        group["y_rot"],
        c=st["color"],
        marker=st["marker"],
        s=st["size"],
        zorder=st["zorder"],
        label=label_map.get(t, t)
    )

# =========================
# 8. Text annotations
# =========================
# 旋转后坐标轴已经改变，所以文字偏移也改用旋转后的 x/y。
# 文字方向保持水平：rotation=0。
for _, row in gdf_rot.iterrows():
    is_main = row["type"] == "main"

    fs = 15 if is_main else 11
    dx = 0.04 if is_main else 0.035
    dy = 0.03 if is_main else 0.025

    if is_main:
        ax.text(
            row["x_rot"] + dx,
            row["y_rot"] - dy,
            row["name_en"],
            fontsize=fs,
            fontname="Times New Roman",
            ha="left",
            va="top",
            rotation=0,
            color="black",
            zorder=6
        )
    else:
        ax.text(
            row["x_rot"] + dx,
            row["y_rot"] + dy,
            row["name_en"],
            fontsize=fs,
            fontname="Times New Roman",
            ha="left",
            va="bottom",
            rotation=0,
            color="black",
            zorder=6
        )

# =========================
# 9. Auto-crop around rotated points
# =========================
xmin, xmax = gdf_rot["x_rot"].min(), gdf_rot["x_rot"].max()
ymin, ymax = gdf_rot["y_rot"].min(), gdf_rot["y_rot"].max()

xpad = (xmax - xmin) * 0.30
ypad = (ymax - ymin) * 0.25

ax.set_xlim(xmin - xpad, xmax + xpad)
ax.set_ylim(ymin - ypad, ymax + ypad)

# =========================
# 10. Beautify
# =========================
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

# =========================
# 11. Legend
# =========================
leg = ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.08, 0.90),
    fontsize=10,
    title="Survey Area Types",
    title_fontsize=11,
    markerscale=0.8,
    frameon=True,
    facecolor="white",
    edgecolor="#999999",
    borderpad=0.6,
    labelspacing=0.5,
    handletextpad=0.6,
    borderaxespad=0.0,
    handlelength=1.2,
)

for t in leg.get_texts():
    t.set_color("black")
    t.set_fontname("Times New Roman")

leg.get_title().set_color("black")
leg.get_title().set_fontname("Times New Roman")

# =========================
# 12. Save
# =========================
plt.tight_layout()
plt.savefig(OUT_PNG, dpi=300, facecolor="white", bbox_inches="tight")
plt.savefig(OUT_PDF, facecolor="white", bbox_inches="tight")
plt.show()

print(f"PNG saved to: {OUT_PNG}")
print(f"PDF saved to: {OUT_PDF}")

name,type,parent,province,query,lon_gcj,lat_gcj,lon,lat,formatted_address,level
当涂,main,,安徽省,安徽省当涂,118.497873,31.570857,118.49265019040642,31.572806479015117,安徽省马鞍山市当涂县,区县
塘南,sub,当涂,安徽省,安徽省当涂塘南,118.660473,31.41392,118.65545317752553,31.41603873704401,安徽省马鞍山市当涂县塘南镇,乡镇
湖阳,sub,当涂,安徽省,安徽省当涂湖阳,118.769985,31.413551,118.76482203485034,31.41554748008119,安徽省马鞍山市当涂县湖阳乡,乡镇
博望,sub,当涂,安徽省,安徽省当涂博望,118.497873,31.570857,118.49265019040642,31.572806479015117,安徽省马鞍山市当涂县,区县
芜湖,main,,安徽省,安徽省芜湖,118.433065,31.352614,118.42767439097202,31.35445884296243,安徽省芜湖市,市
六郎,sub,芜湖,安徽省,安徽省芜湖六郎,118.546331,31.245587,118.5413056868538,31.247757333446142,安徽省芜湖市湾沚区六郎镇,乡镇
池州,main,,安徽省,安徽省池州,117.495663,30.674264,117.49019034291115,30.676615197481233,安徽省池州市,市
茅坦,sub,池州,安徽省,安徽省池州茅坦,117.498887,30.438252,117.49344652984195,30.440696639743262,安徽省池州市贵池区茅坦,村庄
双湖,sub,池州,安徽省,安徽省池州双湖,117.737094,30.723999,117.73168143078706,30.726358473416106,安徽省池州市贵池区双湖村,村庄
马衙,sub,池州,安徽省,安徽省池州马衙,117.619186,30.63818,117.61395974569325,30.640754756324803,安徽省池州市贵池区马衙街道,乡镇
刘街,sub,池州,安徽省,安徽省池州刘街,117.663669,30.439536,117.65839670608499,30.442117293820033,安徽省池州市贵池区刘街街道,乡镇
石门,sub,池州,安徽省,安徽省池州石门,117.537484,30.442438,117.53217099922676,30.44499351144932,安徽省池州市贵池区石门,村庄
青阳,main,,安徽省,安徽省青阳,117.847366,30.639006,117.84212933622237,30.64151795203161,安徽省池州市青阳县,区县
酉华,sub,青阳,安徽省,安徽省青阳酉华,117.866472,30.631143,117.86128053067405,30.633690041140724,安徽省池州市青阳县酉华街道,乡镇
陵阳,sub,青阳,安徽省,安徽省青阳陵阳,117.900834,30.418648,117.89571196613973,30.421295476673794,安徽省池州市青阳县陵阳街道,乡镇
新河,sub,青阳,安徽省,安徽省青阳新河,117.904979,30.673042,117.89983751692499,30.675602345056717,安徽省池州市青阳县新河镇,乡镇
石台,main,,安徽省,安徽省石台,117.486211,30.210218,117.4807488439606,30.21265713211787,安徽省池州市石台县,区县
横渡,sub,石台,安徽省,安徽省石台横渡,117.511111,30.203013,117.50573327057927,30.205526670056315,安徽省池州市石台县横渡,住宅区
七都,sub,石台,安徽省,安徽省石台七都,117.77922,30.233542,117.77386834723771,30.236043412223275,安徽省池州市石台县七都镇,乡镇
铜陵,main,,安徽省,安徽省铜陵,117.811298,30.945214,117.8059431928701,30.94749602743858,安徽省铜陵市,市
西湖,sub,铜陵,安徽省,安徽省铜陵西湖,117.852917,30.979763,117.84766374922515,30.982100883363557,安徽省铜陵市铜官区西湖,住宅区
钟鸣,sub,铜陵,安徽省,安徽省铜陵钟鸣,118.053196,30.975725,118.04773153389426,30.977802360684798,安徽省铜陵市义安区钟鸣镇,乡镇
宣城,main,,安徽省,安徽省宣城,118.759127,30.939278,118.7540026242585,30.941496463365755,安徽省宣城市,市
杨柳,sub,宣城,安徽省,安徽省宣城杨柳,118.894967,31.167723,118.89002119233785,31.169992050017527,安徽省宣城市宣州区杨柳,村庄
泾县,main,,安徽省,安徽省泾县,118.419552,30.688793,118.41420497373738,30.69097178931369,安徽省宣城市泾县,区县
榔桥,sub,泾县,安徽省,安徽省泾县榔桥,118.44915,30.475889,118.4438707580898,30.478183884465388,安徽省宣城市泾县榔桥,村庄
云岭,sub,泾县,安徽省,安徽省泾县云岭,118.22765,30.603676,118.2224351149972,30.60606274501022,安徽省宣城市泾县云岭,住宅区
桃花潭,sub,泾县,安徽省,安徽省泾县桃花潭,118.145926,30.478483,118.14052805772255,30.480778002999582,安徽省宣城市泾县桃花潭,村庄
茂林,sub,泾县,安徽省,安徽省泾县茂林,,,,,,
黄村,sub,泾县,安徽省,安徽省泾县黄村,118.402596,30.67573,118.3972421174521,30.67791240061666,安徽省宣城市泾县黄村,村庄
繁昌,main,,安徽省,安徽省繁昌,118.198536,31.101766,118.19321619791425,31.103843494407386,安徽省芜湖市繁昌区,区县
繁阳,sub,繁昌,安徽省,安徽省繁昌繁阳,118.194259,31.096749,118.18892894766265,31.098821797807933,安徽省芜湖市繁昌区繁阳镇,乡镇
新港,sub,繁昌,安徽省,安徽省繁昌新港,118.077121,31.165741,118.07160428436646,31.167671697484927,安徽省芜湖市繁昌区新港镇,乡镇
孙村,sub,繁昌,安徽省,安徽省繁昌孙村,118.116782,31.05262,118.11128135741087,31.054600033661625,安徽省芜湖市繁昌区孙村,村庄
南陵,main,,安徽省,安徽省南陵,118.334083,30.914621,118.32878859087863,30.91676441211477,安徽省芜湖市南陵县,区县
许镇,sub,南陵,安徽省,安徽省南陵许镇,118.410581,31.062227,118.4051940533235,31.064193647066343,安徽省芜湖市南陵县许镇,乡镇
何湾,sub,南陵,安徽省,安徽省南陵何湾,118.371345,30.987762,118.36598285192018,30.989798197887765,安徽省芜湖市南陵县何湾,村庄
弋江,sub,南陵,安徽省,安徽省南陵弋江,118.474476,30.90395,118.46923077980729,30.906104143076902,安徽省芜湖市南陵县弋江镇,乡镇
籍山,sub,南陵,安徽省,安徽省南陵籍山,118.327852,30.912093,118.32256925415706,30.914249773017943,安徽省芜湖市南陵县籍山,住宅区
黄山,main,,安徽省,安徽省黄山,118.337643,29.714886,118.33244734523981,29.71729366451174,安徽省黄山市,市
焦村,sub,黄山,安徽省,安徽省黄山焦村,118.079913,30.178958,118.07448362632009,30.181274656200245,安徽省黄山市黄山区焦村,村庄
乌石,sub,黄山,安徽省,安徽省黄山乌石,118.189233,29.806973,118.18400444871908,29.80940072212733,安徽省黄山市休宁县乌石,村庄
新明,sub,黄山,安徽省,安徽省黄山新明,118.264767,30.347572,118.25960454520543,30.35004168356102,安徽省黄山市黄山区新明乡,乡镇
甘棠,sub,黄山,安徽省,安徽省黄山甘棠,118.130529,30.287607,118.12511758746324,30.289918208765812,安徽省黄山市黄山区甘棠镇,乡镇
新丰,sub,黄山,安徽省,安徽省黄山新丰,118.084237,30.501539,118.07877602667428,30.503801033266072,安徽省黄山市黄山区新丰乡,乡镇
宁国,main,,安徽省,安徽省宁国,118.983085,30.634032,118.97803945969947,30.63647422518181,安徽省宣城市宁国市,区县
南极,sub,宁国,安徽省,安徽省宁国南极,119.075656,30.398902,119.07035248719505,30.40118468839456,安徽省宣城市宁国市南极乡,乡镇
高淳,main,,江苏省,江苏省高淳,118.892074,31.328678,118.88711248754704,31.33087489958117,江苏省南京市高淳区,区县
砖墙,sub,高淳,江苏省,江苏省高淳砖墙,118.825414,31.262967,118.82034798184299,31.265089875978905,江苏省南京市高淳区砖墙,村庄
淳溪,sub,高淳,江苏省,江苏省高淳淳溪,118.886021,31.327589,118.88105505070963,31.329781887904733,江苏省南京市高淳区淳溪,住宅区
溧水,main,,江苏省,江苏省溧水,119.028414,31.651108,119.02311719723672,31.653002323886813,江苏省南京市溧水区,区县
石湫,sub,溧水,江苏省,江苏省溧水石湫,118.910885,31.637562,118.90589906532877,31.6397107513481,江苏省南京市溧水区石湫,住宅区
